# Notebook 02: Data Cleaning & Feature Engineering

## Purpose

Transform raw DataCo dataset into a clean, analysis-ready format based on findings from Notebook 01.

## Cleaning Plan

1. **Drop columns** — redundant, null, or non-analytical
2. **Rename columns** — convert to snake_case
3. **Fix data types** — convert date columns
4. **Handle nulls** — contextual decisions per column
5. **Feature engineering** — create derived columns for analysis
6. **Validate** — sanity checks before saving
7. **Save** — output to `data/processed/dataco_cleaned.csv`

In [3]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Load raw data
df = pd.read_csv(
    '../data/raw/DataCoSupplyChainDataset.csv',
    encoding='ISO-8859-1'
)

print(f"✅ Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Raw data loaded: 180,519 rows × 53 columns


### 1. Dropping Redundant Columns

 Following GDPR/CCPA principles by default — dropping PII when not needed, using surrogate IDs for tracking, and aggregating data before sharing with stakeholders

In [4]:
cols_to_drop = [
    # Duplicate columns (verified with .equals() in Notebook 01)
    'Order Customer Id',        # duplicate of Customer Id
    'Order Item Total',         # duplicate of Sales per customer
    'Order Profit Per Order',   # duplicate of Benefit per order
    'Order Item Cardprod Id',   # duplicate of Product Card Id
    'Product Category Id',      # duplicate of Category Id
    'Order Item Product Price', # duplicate of Product Price
    
    # No analytical value
    'Product Description',      # 100% null
    'Product Image',            # URL, not useful
    #Remove all presonally identifiable information (PII) columns
    'Customer Password',        # anonymized
    'Customer Email',           # anonymized
    'Customer Street',          # anonymized
    'Customer Fname',           # anonymized (XXXXX)
    'Customer Lname',           # anonymized
]

df_clean = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Dropped 13 columns
Shape: 180,519 rows × 40 columns


### 2. Renaming to Snake Case

In [5]:
# Rename to snake_case
rename_map = {
    'Type': 'payment_type',
    'Days for shipping (real)': 'days_shipping_real',
    'Days for shipment (scheduled)': 'days_shipping_scheduled',
    'Benefit per order': 'profit_per_order',
    'Sales per customer': 'sales_per_customer',
    'Delivery Status': 'delivery_status',
    'Late_delivery_risk': 'late_delivery_risk',
    'Category Id': 'category_id',
    'Category Name': 'category_name',
    'Customer City': 'customer_city',
    'Customer Country': 'customer_country',
    'Customer Id': 'customer_id',
    'Customer Segment': 'customer_segment',
    'Customer State': 'customer_state',
    'Customer Zipcode': 'customer_zipcode',
    'Department Id': 'department_id',
    'Department Name': 'department_name',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Market': 'market',
    'Order City': 'order_city',
    'Order Country': 'order_country',
    'order date (DateOrders)': 'order_date',
    'Order Id': 'order_id',
    'Order Item Discount': 'order_item_discount',
    'Order Item Discount Rate': 'order_item_discount_rate',
    'Order Item Id': 'order_item_id',
    'Order Item Profit Ratio': 'order_item_profit_ratio',
    'Order Item Quantity': 'order_item_quantity',
    'Sales': 'sales',
    'Order Region': 'order_region',
    'Order State': 'order_state',
    'Order Status': 'order_status',
    'Order Zipcode': 'order_zipcode',
    'Product Card Id': 'product_card_id',
    'Product Name': 'product_name',
    'Product Price': 'product_price',
    'Product Status': 'product_status',
    'shipping date (DateOrders)': 'shipping_date',
    'Shipping Mode': 'shipping_mode',
}

df_clean = df_clean.rename(columns=rename_map)

print(f"Renamed {len(rename_map)} columns ✅")
print(f"New columns sample: {list(df_clean.columns)}")

Renamed 40 columns ✅
New columns sample: ['payment_type', 'days_shipping_real', 'days_shipping_scheduled', 'profit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']


### 3. Fix Data Types

Convert `shipping_date` to datetime. `order_date` is already datetime.

In [6]:
#recheck the dtypes
df_clean[['order_date','shipping_date']].dtypes

order_date       str
shipping_date    str
dtype: object

In [7]:
# Convert to datetime
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping_date'])

df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])

# Verify
print(f"order_date dtype: {df_clean['order_date'].dtype}")
print(f"shipping_date dtype: {df_clean['shipping_date'].dtype}")
print(f"\nDate ranges:")
print(f"  Orders: {df_clean['order_date'].min()} → {df_clean['order_date'].max()}")
print(f"  Shipping: {df_clean['shipping_date'].min()} → {df_clean['shipping_date'].max()}")

order_date dtype: datetime64[us]
shipping_date dtype: datetime64[us]

Date ranges:
  Orders: 2015-01-01 00:00:00 → 2018-01-31 23:38:00
  Shipping: 2015-01-03 00:00:00 → 2018-02-06 22:14:00


### 4. Hande Null Values

In [8]:
# Check remaining nulls
nulls = df_clean.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

if len(nulls) > 0:
    print("Columns with nulls:")
    for col, count in nulls.items():
        pct = (count / len(df_clean)) * 100
        print(f"  {col:25s} {count:>8,} ({pct:>5.2f}%)")
else:
    print("No nulls!")

Columns with nulls:
  order_zipcode              155,679 (86.24%)
  customer_zipcode                 3 ( 0.00%)


In [17]:
# Drop order_zipcode (86% null, not useful)
df_clean = df_clean.drop(columns=['order_zipcode'])

# Fill customer_zipcode nulls with 0 (minor, preserve rows)
df_clean['customer_zipcode'] = df_clean['customer_zipcode'].fillna(0).astype(int)

# Verify
print(f"Shape after null handling: {df_clean.shape}")
print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")

Shape after null handling: (180519, 39)
Remaining nulls: 0


In [10]:
df_clean.head(10)

,payment_type,days_shipping_real,days_shipping_scheduled,profit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_id,customer_segment,customer_state,customer_zipcode,department_id,department_name,latitude,longitude,market,order_city,order_country,order_date,order_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_profit_ratio,order_item_quantity,sales,order_region,order_state,order_status,order_zipcode,product_card_id,product_name,product_price,product_status,shipping_date,shipping_mode
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,20755,Consumer,PR,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Bekasi,Indonesia,2018-01-31 22:56:00,77202,13.11,0.04,180517,0.29,1,327.75,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,19492,Consumer,PR,725.00,2,Fitness,18.28,-66.04,Pacific Asia,Bikaner,India,2018-01-13 12:27:00,75939,16.39,0.05,179254,-0.80,1,327.75,South Asia,Rajastán,PENDING,NaN,1360,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.78,309.72,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,19491,Consumer,CA,"95,125.00",2,Fitness,37.29,-121.88,Pacific Asia,Bikaner,India,2018-01-13 12:06:00,75938,18.03,0.06,179253,-0.80,1,327.75,South Asia,Rajastán,CLOSED,NaN,1360,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,19490,Home Office,CA,"90,027.00",2,Fitness,34.13,-118.29,Pacific Asia,Townsville,Australia,2018-01-13 11:45:00,75937,22.94,0.07,179252,0.08,1,327.75,Oceania,Queensland,COMPLETE,NaN,1360,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,19489,Corporate,PR,725.00,2,Fitness,18.25,-66.04,Pacific Asia,Townsville,Australia,2018-01-13 11:24:00,75936,29.50,0.09,179251,0.45,1,327.75,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class
5,TRANSFER,6,4,18.58,294.98,Shipping canceled,0,73,Sporting Goods,Tonawanda,EE. UU.,19488,Consumer,NY,"14,150.00",2,Fitness,43.01,-78.88,Pacific Asia,Toowoomba,Australia,2018-01-13 11:03:00,75935,32.78,0.10,179250,0.06,1,327.75,Oceania,Queensland,CANCELED,NaN,1360,Smart watch,327.75,0,2018-01-19 11:03:00,Standard Class
6,DEBIT,2,1,95.18,288.42,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,19487,Home Office,PR,725.00,2,Fitness,18.24,-66.04,Pacific Asia,Guangzhou,China,2018-01-13 10:42:00,75934,39.33,0.12,179249,0.33,1,327.75,Eastern Asia,Guangdong,COMPLETE,NaN,1360,Smart watch,327.75,0,2018-01-15 10:42:00,First Class
7,TRANSFER,2,1,68.43,285.14,Late delivery,1,73,Sporting Goods,Miami,EE. UU.,19486,Corporate,FL,"33,162.00",2,Fitness,25.93,-80.16,Pacific Asia,Guangzhou,China,2018-01-13 10:21:00,75933,42.61,0.13,179248,0.24,1,327.75,Eastern Asia,Guangdong,PROCESSING,NaN,1360,Smart watch,327.75,0,2018-01-15 10:21:00,First Class
8,CASH,3,2,133.72,278.59,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,19485,Corporate,PR,725.00,2,Fitness,18.23,-66.04,Pacific Asia,Guangzhou,China,2018-01-13 10:00:00,75932,49.16,0.15,179247,0.48,1,327.75,Eastern Asia,Guangdong,CLOSED,NaN,1360,Smart watch,327.75,0,2018-01-16 10:00:00,Second Class
9,CASH,2,1,132.15,275.31,Late delivery,1,73,Sporting Goods,San Ramon,EE. UU.,19484,Corporate,CA,"94,583.00",2,Fitness,37.77,-121.97,Pacific Asia,Guangzhou,China,2018-01-13 09:39:00,75931,52.44,0.16,179246,0.48,1,327.75,Eastern Asia,Guangdong,CLOSED,NaN,1360,Smart watch,327.75,0,2018-01-15 09:39:00,First Class


In [14]:
late_delivery_unique = df_clean['late_delivery_risk'].unique()
print(f"Unique values in 'late_delivery_risk': {late_delivery_unique}")

unique_values_in_all_categorical_cols = df_clean.select_dtypes(include='object').columns
for col in unique_values_in_all_categorical_cols:
    unique_vals = df_clean[col].unique()
    print(f"Unique values in '{col}': {unique_vals}")

Unique values in 'late_delivery_risk': [0 1]
Unique values in 'payment_type': <StringArray>
['DEBIT', 'TRANSFER', 'CASH', 'PAYMENT']
Length: 4, dtype: str
Unique values in 'delivery_status': <StringArray>
['Advance shipping', 'Late delivery', 'Shipping on time', 'Shipping canceled']
Length: 4, dtype: str
Unique values in 'category_name': <StringArray>
[      'Sporting Goods',               'Cleats',        'Shop By Sport',      'Women's Apparel',          'Electronics',         'Boxing & MMA',     'Cardio Equipment',             'Trade-In',
     'Kids' Golf Clubs',   'Hunting & Shooting',  'Baseball & Softball',       'Men's Footwear',     'Camping & Hiking', 'Consumer Electronics',             'Cameras ',            'Computers',
           'Basketball',               'Soccer',       'Girls' Apparel',          'Accessories',     'Women's Clothing',               'Crafts',       'Men's Clothing',     'Tennis & Racquet',
  'Fitness Accessories',      'As Seen on  TV!',           'Golf Ba

C:\Users\nikhi\AppData\Local\Temp\ipykernel_33892\2640121198.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  unique_values_in_all_categorical_cols = df_clean.select_dtypes(include='object').columns


## 5. Feature Engineering

Create derived columns from existing data. These will be essential for analysis and dashboards.

| New Column | Derived From | Purpose |
|------------|--------------|---------|
| `is_late` | `late_delivery_risk` | Boolean for cleaner filtering |
| `shipping_delay_days` | `days_shipping_real - days_shipping_scheduled` | Quantify lateness |
| `profit_margin_pct` | `profit_per_order / sales` | % margin per order |
| `order_year`, `order_month`, `order_day_of_week` | `order_date` | Time-based analysis |
| `order_year_month` | `order_date` | Monthly trends |
| `is_profitable` | `profit_per_order > 0` | Quick loss filter |

# ============================================================
# FEATURE ENGINEERING
# ============================================================

# 1. Binary late flag (cleaner than late_delivery_risk)
df_clean['is_late'] = df_clean['late_delivery_risk'].astype(bool)

# 2. Shipping delay in days (negative = early, 0 = on time, positive = late)
df_clean['shipping_delay_days'] = (
    df_clean['days_shipping_real'] - df_clean['days_shipping_scheduled']
)

# 3. Profit margin %
df_clean['profit_margin_pct'] = (
    df_clean['profit_per_order'] / df_clean['sales'] * 100
).round(2)

# 4. Time-based features from order_date
df_clean['order_year'] = df_clean['order_date'].dt.year
df_clean['order_month'] = df_clean['order_date'].dt.month
df_clean['order_day_of_week'] = df_clean['order_date'].dt.day_name()
df_clean['order_year_month'] = df_clean['order_date'].dt.to_period('M').astype(str)

# 5. Profitability flag
df_clean['is_profitable'] = df_clean['profit_per_order'] > 0

# Verify
print(f"Shape: {df_clean.shape}")
print(f"\nNew columns sample:")
print(df_clean[['is_late', 'shipping_delay_days', 'profit_margin_pct', 
                'order_year', 'order_month', 'order_year_month', 'is_profitable']].head())